The cube is too large to load into memory. These cells use Dask reductions, preserve the native Zarr chunks, and return small tables.
## Cell 2 — Start a small Dask cluster for 4 vCPU / 12 GB RAM

Use 2 workers × 2 threads so each worker has enough memory for the cube’s large native chunks.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import xarray as xr
import dask
import dask.array as da
from dask.distributed import Client, LocalCluster

dask.config.set({
    "array.slicing.split_large_chunks": False,
    "distributed.worker.memory.target": 0.65,
    "distributed.worker.memory.spill": 0.75,
    "distributed.worker.memory.pause": 0.85,
})

cluster = LocalCluster(
    n_workers=2,
    threads_per_worker=2,
    memory_limit="4.0GB",
    dashboard_address=None,
)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 4,Total memory: 7.45 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39691,Workers: 2
Dashboard: http://127.0.0.1:8787/status,Total threads: 4
Started: Just now,Total memory: 7.45 GiB
Comm: tcp://127.0.0.1:35473,Total threads: 2
Dashboard: http://127.0.0.1:38427/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:35705,


2026-06-19 17:11:09,626 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 14b01100bc0f82f8e7ac11e8f8f24887 initialized by task ('rechunk-merge-rechunk-transfer-2e44a6b487d65cfe9542288228b85c2b', 3, 0, 3, 1) executed on worker tcp://127.0.0.1:39613
2026-06-19 17:11:11,626 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle fb90030fbe2f1083a184bcc743c80f01 initialized by task ('rechunk-merge-rechunk-transfer-2e44a6b487d65cfe9542288228b85c2b', 2, 0, 2, 3) executed on worker tcp://127.0.0.1:39613
2026-06-19 17:11:14,507 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 14b01100bc0f82f8e7ac11e8f8f24887 deactivated due to stimulus 'task-finished-1781885474.5054066'
2026-06-19 17:11:16,123 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle cfe24f7cc27c2088b0e3c5697a1df3b7 initialized by task ('rechunk-merge-rechunk-transfer-2e44a6b487d65cfe9542288228b85c2b', 4, 0, 4, 2) executed on worker tcp://127.0.0.1:39613
2026-06-19 17:11:17,151 - distributed.shuffle.

## Cell 3 — Open the cube



In [2]:
COMBINED_URL = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/antarctica-combined.zarr"
ICETEMP_URL = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/icetemp.zarr"
SEC_URL = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/sec.zarr"
CALVING_URL = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/icemask_composite.zarr"
VELOCITY_URL = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/ice_velocity.zarr"

def open_remote_zarr(url: str) -> xr.Dataset:
    kwargs = dict(consolidated=True, chunks={})
    try:
        return xr.open_zarr(url, create_default_indexes=False, **kwargs)
    except TypeError:
        return xr.open_zarr(url, **kwargs)


combined_full = open_remote_zarr(COMBINED_URL)
icetemp_full = open_remote_zarr(ICETEMP_URL)
sec_full = open_remote_zarr(SEC_URL)
calving_full = open_remote_zarr(CALVING_URL)
velocity_full = open_remote_zarr(VELOCITY_URL)


ds = xr.merge([
    combined_full,
    icetemp_full,
    sec_full,
    calving_full,
    velocity_full], compat='override')
ds

<xarray.Dataset> Size: 4TB
Dimensions:                                       (y: 49158, x: 57358,
                                                   depth: 91, time_period: 27,
                                                   time: 24)
Coordinates:
    spatial_ref                                   int64 8B ...
    x                                             (x) float32 229kB dask.array<chunksize=(57358,), meta=np.ndarray>
    y                                             (y) float32 197kB dask.array<chunksize=(49158,), meta=np.ndarray>
    depth                                         (depth) int16 182B dask.array<chunksize=(91,), meta=np.ndarray>
    time_period                                   (time_period) int64 216B dask.array<chunksize=(27,), meta=np.ndarray>
    time                                          (time) datetime64[ns] 192B dask.array<chunksize=(24,), meta=np.ndarray>
Data variables: (12/40)
    bedrock_topography_bed                        (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_dataid                     (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_errbed                     (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_firn                       (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_geoid                      (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_mask                       (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    ...                                            ...
    ice_sheet_surface_velocity_easting_stddev     (time, y, x) float32 271GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_magnitude          (time, y, x) float32 271GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_measurement_count  (time, y, x) float32 271GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_northing           (time, y, x) float32 271GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_northing_stddev    (time, y, x) float32 271GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_vertical           (time, y, x) float32 271GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>

In [3]:
dx = abs((ds.x.isel(x=1) - ds.x.isel(x=0)).load().item())
dy = abs((ds.y.isel(y=1) - ds.y.isel(y=0)).load().item())
km2 = dx * dy / 1e6

ds[[
    "bedrock_topography_mask",
    "bedrock_topography_bed",
    "bedrock_topography_thickness",
    "bedrock_topography_errbed",
    "ice_shelf_basal_melt_rate",
    "groundlines_mask",
    "subglacial_lakes_mask",
    "supra_glacial_lakes_mask",
]]

<xarray.Dataset> Size: 65GB
Dimensions:                       (y: 49158, x: 57358)
Coordinates:
    spatial_ref                   int64 8B ...
    x                             (x) float32 229kB dask.array<chunksize=(57358,), meta=np.ndarray>
    y                             (y) float32 197kB dask.array<chunksize=(49158,), meta=np.ndarray>
Data variables:
    bedrock_topography_mask       (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_bed        (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_thickness  (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_errbed     (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    ice_shelf_basal_melt_rate     (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    groundlines_mask              (y, x) uint8 3GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    subglacial_lakes_mask         (y, x) uint8 3GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    supra_glacial_lakes_mask      (y, x) uint8 3GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>

## Cell 4 — Area of the main surface classes

Mask codes: 0 ocean, 1 ice-free land, 2 grounded ice, 3 floating ice, 4 Lake Vostok.

In [4]:
mask = ds.bedrock_topography_mask.data
mask_i = da.where(da.isfinite(mask), mask, 255).astype("uint16")

mask_counts = da.bincount(mask_i.ravel(), minlength=256)[:5].compute()

mask_table = pd.DataFrame({
    "code": [0, 1, 2, 3, 4],
    "class": ["ocean", "ice_free_land", "grounded_ice", "floating_ice", "lake_vostok"],
    "cells": mask_counts.astype("int64"),
    "projected_area_km2": mask_counts * km2,
})

mask_table

,code,class,cells,projected_area_km2
0,0,ocean,1460261014,14602610.14
1,1,ice_free_land,7002425,70024.25
2,2,grounded_ice,1199100525,11991005.25
3,3,floating_ice,151720200,1517202.00
4,4,lake_vostok,1520400,15204.00


## Cell 5 — Phenomenon 1: marine-based grounded ice exposure

This estimates how much grounded ice sits on bed below sea level, including deep basins below -1000 m.

In [5]:
bed = ds.bedrock_topography_bed.data
thick = ds.bedrock_topography_thickness.data
err = ds.bedrock_topography_errbed.data

grounded = mask_i == 2
floating = mask_i == 3

marine = grounded & da.isfinite(bed) & (bed < 0)
deep_marine = marine & (bed < -1000)

marine_vals = dask.compute(
    marine.sum() * km2,
    deep_marine.sum() * km2,
    da.nanmean(da.where(marine, bed, np.nan)),
    da.nanmean(da.where(marine, thick, np.nan)),
    da.nanmean(da.where(marine, err, np.nan)),
)

marine_table = pd.DataFrame({
    "metric": [
        "grounded_ice_bed_below_0m_km2",
        "grounded_ice_bed_below_minus_1000m_km2",
        "mean_bed_elevation_m",
        "mean_ice_thickness_m",
        "mean_bed_error_m",
    ],
    "value": marine_vals,
})

marine_table

,metric,value
0,grounded_ice_bed_below_0m_km2,5.457388e+06
1,grounded_ice_bed_below_minus_1000m_km2,5.941695e+05
2,mean_bed_elevation_m,-5.014638e+02
3,mean_ice_thickness_m,2.241388e+03
4,mean_bed_error_m,1.246957e+02


## Cell 6 — Phenomenon 2: floating-ice basal melt and a 5 km grounding-zone belt

The melt variable is labelled as a basal melt rate, but the metadata unit is `m`. If the source convention is m/yr, the volume columns are km³/yr.

In [6]:
from scipy.ndimage import binary_dilation

melt = ds.ice_shelf_basal_melt_rate.data
melt_ok = floating & da.isfinite(melt)

gl = ds.groundlines_mask.data.astype(bool)
gl5 = da.map_overlap(
    binary_dilation,
    gl,
    depth=5,
    boundary=False,
    dtype=bool,
    structure=np.ones((11, 11), bool),
)

grounding_zone = gl5 & floating & da.isfinite(melt)

melt_vals = dask.compute(
    melt_ok.sum() * km2,
    da.nanmean(da.where(melt_ok, melt, np.nan)),
    da.nansum(da.where(melt_ok, melt, 0)) * km2 / 1000,
    (melt_ok & (melt > 0)).sum() * km2,
    da.nansum(da.where(melt_ok & (melt > 0), melt, 0)) * km2 / 1000,
    grounding_zone.sum() * km2,
    da.nanmean(da.where(grounding_zone, melt, np.nan)),
    da.nanmean(da.where(grounding_zone, bed, np.nan)),
    da.nanmean(da.where(grounding_zone, thick, np.nan)),
    da.nanmean(da.where(grounding_zone, err, np.nan)),
)

melt_table = pd.DataFrame({
    "metric": [
        "floating_ice_with_melt_data_km2",
        "floating_ice_mean_melt_m",
        "floating_ice_net_melt_km3_per_dataset_unit",
        "floating_ice_positive_melt_area_km2",
        "floating_ice_positive_melt_km3_per_dataset_unit",
        "grounding_zone_5km_area_km2",
        "grounding_zone_5km_mean_melt_m",
        "grounding_zone_5km_mean_bed_m",
        "grounding_zone_5km_mean_thickness_m",
        "grounding_zone_5km_mean_bed_error_m",
    ],
    "value": melt_vals,
})

melt_table

,metric,value
0,floating_ice_with_melt_data_km2,1.458565e+06
1,floating_ice_mean_melt_m,7.076098e-01
2,floating_ice_net_melt_km3_per_dataset_unit,1.032095e+03
3,floating_ice_positive_melt_area_km2,8.583990e+05
4,floating_ice_positive_melt_km3_per_dataset_unit,1.354090e+03
5,grounding_zone_5km_area_km2,2.548982e+04
6,grounding_zone_5km_mean_melt_m,4.801201e+00
7,grounding_zone_5km_mean_bed_m,-5.308877e+02
8,grounding_zone_5km_mean_thickness_m,5.238826e+02
9,grounding_zone_5km_mean_bed_error_m,4.541095e+01
